In [5]:
import sys
sys.path.append('../..')

import pandas as pd
from utils.db_utils import write_table, read_table

In [6]:
date = read_table("select * from sc_gold.dim_date")
skill_level = read_table("select * from sc_gold.dim_skilllevel")

In [7]:
df = read_table("select * from sc_bronze.dosm_emp_graduates_skill")
df 

,year,skill_level,total_emp_graduate
0,2016,Skilled,2679800.0
1,2016,Semi-skilled,768600.0
2,2016,Low-skilled,28500.0
3,2017,Skilled,2777600.0
4,2017,Semi-skilled,865500.0
5,2017,Low-skilled,36900.0
6,2018,Skilled,2933700.0
7,2018,Semi-skilled,994100.0
8,2018,Low-skilled,46200.0
9,2019,Skilled,3112300.0


In [8]:
df["date"] = pd.to_datetime(df["year"], format="%Y")
df = df.merge(
    date[["date", "date_id"]],
    on="date",
    how="left"
)

df = df.merge(
    skill_level[["skill_level", "skill_level_id"]],
    on="skill_level",
    how="left"
)


df_final = df.drop(columns=["year", "date", "skill_level"])
id_cols = ["date_id", "skill_level_id"]
df_final = df_final[id_cols + [col for col in df_final.columns if col not in id_cols]]

In [9]:
df_final["ds_id"] = ["DS" + str(i+1).zfill(4) for i in range(len(df_final))]
df_final = df_final[["ds_id"] + [c for c in df_final.columns if c != "ds_id"]]
df_final

,ds_id,date_id,skill_level_id,total_emp_graduate
0,DS0001,DT001,SL003,2679800.0
1,DS0002,DT001,SL002,768600.0
2,DS0003,DT001,SL001,28500.0
3,DS0004,DT005,SL003,2777600.0
4,DS0005,DT005,SL002,865500.0
5,DS0006,DT005,SL001,36900.0
6,DS0007,DT009,SL003,2933700.0
7,DS0008,DT009,SL002,994100.0
8,DS0009,DT009,SL001,46200.0
9,DS0010,DT013,SL003,3112300.0


In [10]:
write_table(df_final, "sc_gold", "fact_date_skill")

Table sc_gold.fact_date_skill written successfully.
